# 02 — Pré-cleanessamento

Limpeza, tratamento de ausentes, encoding e normalização/padronização.

In [1]:
# Imports
import pandas as pd
import numpy as np

In [2]:
df = pd.read_parquet("../data/df_model_raw.parquet") # Retorna um DataFrame

In [3]:
df.shape

(724397, 17)

In [4]:
df

,IDADEMAE,ESCMAE2010,ESTCIVMAE,RACACORMAE,QTDGESTANT,QTDPARTNOR,QTDPARTCES,QTDFILVIVO,QTDFILMORT,GRAVIDEZ,MESPRENAT,CONSPRENAT,SEXO,IDADEPAI,latitude,longitude,PREMATURO
0,30,5.0,1.0,1,1.0,0.0,1.0,1.0,0.0,1.0,1.0,11.0,1,39.0,-19.9102,-43.9266,0
1,27,3.0,1.0,1,0.0,0.0,0.0,0.0,0.0,1.0,2.0,8.0,2,39.0,-19.9102,-43.9266,0
2,41,5.0,1.0,4,0.0,0.0,0.0,0.0,0.0,1.0,1.0,9.0,2,36.0,-19.9102,-43.9266,0
3,29,3.0,1.0,1,0.0,0.0,0.0,0.0,0.0,1.0,1.0,9.0,2,22.0,-19.9102,-43.9266,0
4,36,5.0,4.0,1,2.0,0.0,0.0,0.0,2.0,1.0,1.0,9.0,2,38.0,-19.9102,-43.9266,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
724392,27,4.0,4.0,4,1.0,0.0,0.0,0.0,1.0,1.0,3.0,11.0,1,28.0,-18.9141,-48.2749,0
724393,26,2.0,1.0,4,2.0,0.0,2.0,2.0,0.0,1.0,2.0,7.0,2,26.0,-18.9141,-48.2749,0
724394,20,3.0,1.0,4,0.0,0.0,0.0,0.0,0.0,1.0,2.0,17.0,1,27.0,-18.9141,-48.2749,1
724395,32,3.0,1.0,4,0.0,0.0,0.0,0.0,0.0,1.0,3.0,7.0,1,45.0,-18.9141,-48.2749,0


## Convertendo Sentinelas (Sentinelas -> NaNs)
O SINASC tem sentinelas que são ignorados pelo sistema, porém contentdo 9,99,999 como códigos a serem ignorados.

Objetivo: Equalizar os dados para que o Pandas consiga comparar as linhas de forma justa na etapa de duplicadas

### Como identificar as colunas que tem sentinela?

#### 1. Técnica da Frequência de Finais
Ao fazer um `value_counts().head(10)`conseguimos o Top 10 valores mais populares por coluna

In [5]:
cols_to_check = df.columns

for col in cols_to_check:
    top_values = df[col].value_counts(normalize=True).head(10)
    print(f"Coluna: {col}")
    print(top_values)
    print("-" * 30)

Coluna: IDADEMAE
IDADEMAE
26    0.050113
25    0.049815
27    0.049720
28    0.049234
29    0.048605
30    0.048537
24    0.048403
31    0.047466
23    0.046988
22    0.046341
Name: proportion, dtype: float64
------------------------------
Coluna: ESCMAE2010
ESCMAE2010
3.0    0.556979
5.0    0.201869
2.0    0.168285
4.0    0.049425
1.0    0.020503
9.0    0.001544
0.0    0.001396
Name: proportion, dtype: float64
------------------------------
Coluna: ESTCIVMAE
ESTCIVMAE
1.0    0.452861
2.0    0.418797
5.0    0.102056
4.0    0.021099
9.0    0.002910
3.0    0.002276
Name: proportion, dtype: float64
------------------------------
Coluna: RACACORMAE
RACACORMAE
4    0.545350
1    0.321133
2    0.098646
     0.025050
3    0.007656
5    0.002165
Name: proportion, dtype: float64
------------------------------
Coluna: QTDGESTANT
QTDGESTANT
0.0    0.389371
1.0    0.314566
2.0    0.168624
3.0    0.072630
4.0    0.030607
5.0    0.012985
6.0    0.005886
7.0    0.002726
8.0    0.001301
9.0    0.00065

### Como identificar o que é "lixo" do que é dado real? Será que todos os 9 na tabela toda são códigos de "ignorado"?

#### 2. Três critérios: Frequência, semântica e descontinuidade

Às vezes pelo nome da tabela essa informação consegue ser óbvia, mas outra vezes conseguimos seguir pelo o que os dados estão dizendo.

1. Onde o 9 e 99 são claramente SENTINELAS (lixo)
    - `MESPRENAT` **(99.0 com 2%)**: É sentinela. Ninguém começa o pré-natal no 99º mês. Note que o 9 aqui não aparece, o erro padrão do SINASC para meses é 99.
    - `ESTCIVMAE` e `ESCMAE2010` **(9.0 com ~0.2%)**: São sentinelas. Estão no final da lista, com frequências baixas, e o dicionário do SINASC reserva o 9 para "Ignorado" em variáveis categóricas codificadas de 1 a 5.
    - `SEXO` **(0 com 0.01%)**: O 0 aqui é sentinela para "Ignorado".

2. Onde o 9 é um VALOR REAL (dado válido)
    - Nestas colunas, o 9 faz parte de uma distribuição que vai diminuindo organicamente.
        - `QTDGESTANT, QTDPARTNOR, QTDFILVIVO`: * Observe a sequência: ...7, 8, 9, 10, 11...
            - O valor 9.0 aparece com uma frequência de **0.000650** (muito pequena). Se fosse um erro do sistema para "ignorado", esse número seria muito mais alto (como os 2% do `MESPRENAT`).
            - Aqui, 9 significa que a mãe realmente teve 9 gestações/partos. É um dado raro, mas biologicamente possível e importante para o modelo (multiparidade é fator de risco).
     
3. O caso especial: `RACACORMAE`
    - Note que em `RACACORMAE` apareceu um valor vazio com **2.5%**.
        - Isso é um "missing" disfarçado de string vazia. Você deve converter isso para NaN antes de qualquer outra coisa.


In [6]:
# Cria uma cópia para limpeza
df_clean = df.copy()

# Sentinelas Categóricos (9 ou 0 significa ignorado)
cols_cat_ignorado = ['ESTCIVMAE', 'ESCMAE2010', 'GRAVIDEZ', 'SEXO']
df_clean[cols_cat_ignorado] = df_clean[cols_cat_ignorado].replace({9: np.nan, 9.0: np.nan, 0: np.nan})

# Sentinelas de Contagem/Tempo (99 significa ignorado)
cols_tempo_ignorado = ['MESPRENAT', 'CONSPRENAT']
df_clean[cols_tempo_ignorado] = df_clean[cols_tempo_ignorado].replace({99: np.nan, 99.0: np.nan})

# Filtro de sanidade para IDADEPAI (valores absurdos de 99, 120 etc)
df_clean.loc[df_clean['IDADEPAI'] > 80, 'IDADEPAI'] = np.nan


## Limpeza de Strings e Tipagem
Algumas strings na base que podem conter sugeira, como por exemplo `"MG"` tendo valores como `"MG "`(contendoespaços) o que é considerado diferente para o computador.

Objetivo: Remover espaços em branco e garantir que as colunas numéricas sejam de fato `float` ou `int`.

In [7]:
# Tratamento de string vazia (ex: RACACORMAE)
df_clean = df_clean.replace(r'^\s*$', np.nan, regex=True)

## Identificação e Remoção de Duplicadas

Após a "normalização" dos dados (sentinelas viraram nulos e espaços foram removidos) conseguimos uma base melhor tratada para identificar duplicadas e removê-las.

Objetivo: Identificar e remover as duplicadas

In [8]:
antes = len(df_clean)
df_clean = df_clean.drop_duplicates()
depois = len(df_clean)

print(f"Registros removidos: {antes - depois}")

Registros removidos: 32015



# PAREI: porque escolhemos a média? Ainda sim, será que "Nao_informado" é uma categoria interessante? Já que já temos um código?
## Identificação e Tratamento de Nulos (imputação)

Agora que removemos as duplicadas precisamos tratar os nulos, isto é, nosso modelo não conseguiria cleanessar valores não numéricos.

In [ ]:
# Exemplo de imputação estratégica:
# Para IDADEPAI e IDADEMAE: Mediana (evita viés de outliers)
df_clean['IDADEPAI'] = df_clean['IDADEPAI'].fillna(df_clean['IDADEPAI'].median())

# Para Categóricas: Criar uma nova categoria "Nao_Informado"
# Isso é importante porque a falta de info é um preditor de risco!
cols_fill_cat = ['ESTCIVMAE', 'ESCMAE2010', 'RACACORMAE']
for col in cols_fill_cat:
    df_clean[col] = df_clean[col].fillna(-1) # ou uma string como '99' para o encoder